# Notebook 01 — EDA y calidad de datos | A/B Testing en AWS

**Objetivo:** auditar la calidad de datos y describir un experimento A/B, sin ejecutar pruebas de hipótesis.

**Entrada:** `s3://TU_BUCKET/raw/ab_data.csv` o archivo local de respaldo. **Salidas:** `processed/ab_data_clean.csv`, `results/eda_summary.csv`, `results/data_quality_report.csv`, `results/daily_conversions.csv` y gráficos PNG.

> **Advertencia metodológica:** esta limpieza construye una muestra *per-protocol* simplificada. Si la asignación original del experimento está disponible, un análisis *intention-to-treat* puede ser más adecuado. No inferir causalidad antes de revisar aleatorización, contaminación y reglas del experimento.

## 1. Configuración y librerías
En SageMaker JupyterLab usa un kernel Python 3. Verifica que tu rol IAM permita `s3:GetObject` y `s3:PutObject` para los objetos apropiados. Cambia `BUCKET`.

In [ ]:
# Descomenta si es necesario:
# %pip install pandas numpy matplotlib seaborn boto3
from io import BytesIO
from pathlib import Path
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

BUCKET = "TU_BUCKET"                  # Reemplaza antes de ejecutar
KEY = "raw/ab_data.csv"
USE_S3 = True                         # False para correr localmente
LOCAL_PATH = "data/ab_data.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3") if USE_S3 else None

## 2. Ingesta reproducible desde S3

In [ ]:
if USE_S3:
    if BUCKET == "TU_BUCKET":
        raise ValueError("Configura BUCKET con el nombre de tu bucket S3")
    obj = s3.get_object(Bucket=BUCKET, Key=KEY)
    df = pd.read_csv(BytesIO(obj["Body"].read()))
    print(f"Leídos {len(df):,} registros de s3://{BUCKET}/{KEY}")
else:
    df = pd.read_csv(LOCAL_PATH)
    print(f"Leídos {len(df):,} registros desde {LOCAL_PATH}")

display(df.head())

## 3. Validación de esquema y tipos

In [ ]:
expected = {"user_id", "timestamp", "group", "landing_page", "converted"}
missing_columns = expected - set(df.columns)
assert not missing_columns, f"Faltan columnas: {missing_columns}"

df = df.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
df["converted"] = pd.to_numeric(df["converted"], errors="coerce")

print("Dimensiones:", df.shape)
print("Tipos:")
print(df.dtypes)
print("Primeras fechas:", df["timestamp"].min(), df["timestamp"].max())
display(df.describe(include="all"))

## 4. Auditoría de valores faltantes y duplicados

In [ ]:
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": (100 * df.isna().mean()).round(3)
})
display(missing)

print("Filas idénticas duplicadas:", int(df.duplicated().sum()))
print("IDs de usuario repetidos (registros extra):", int(df.duplicated("user_id").sum()))
print("Usuarios únicos:", df["user_id"].nunique())

multiple_users = df.loc[df.duplicated("user_id", keep=False)].sort_values(["user_id", "timestamp"])
display(multiple_users.head(12))

## 5. Consistencia del experimento
El grupo control debe mostrar `old_page` y el tratamiento `new_page`. La distribución cruda puede contener asignaciones no concordantes.

In [ ]:
display(pd.crosstab(df["group"], df["landing_page"], margins=True))
valid_group = df["group"].isin(["control", "treatment"])
valid_page = df["landing_page"].isin(["old_page", "new_page"])
valid_convert = df["converted"].isin([0, 1])
valid_user = df["user_id"].notna()
valid_time = df["timestamp"].notna()
assignment_match = (
    ((df["group"] == "control") & (df["landing_page"] == "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
)
flags = pd.DataFrame({
    "invalid_group": ~valid_group,
    "invalid_page": ~valid_page,
    "invalid_conversion": ~valid_convert,
    "missing_user_id": ~valid_user,
    "invalid_timestamp": ~valid_time,
    "assignment_mismatch": valid_group & valid_page & ~assignment_match,
    "repeated_user_row": df.duplicated("user_id", keep=False)
})
quality_report = pd.DataFrame({
    "issue": flags.columns,
    "rows_affected": flags.sum().values,
    "pct_of_rows": (flags.mean().values * 100).round(3)
})
display(quality_report)

print("Asignación no concordante por grupo (%):")
display(pd.crosstab(df["group"], flags["assignment_mismatch"], normalize="index").mul(100).round(2))

## 6. Distribución de tráfico y conversiones, antes de limpiar

In [ ]:
raw_summary = df[df["group"].isin(["control", "treatment"])].groupby("group").agg(
    registros=("user_id", "size"),
    usuarios_unicos=("user_id", "nunique"),
    tasa_conversion_cruda=("converted", "mean")
)
display(raw_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df[df["group"].isin(["control", "treatment"])], x="group", order=["control", "treatment"], ax=axes[0])
axes[0].set(title="Distribución cruda de registros", xlabel="Grupo", ylabel="Registros")
sns.countplot(data=df[df["converted"].isin([0, 1])], x="converted", ax=axes[1])
axes[1].set(title="Conversiones y no conversiones", xlabel="Convirtió", ylabel="Registros")
plt.tight_layout(); plt.show()

## 7. Limpieza documentada
Conservamos una observación válida por `user_id`, la más temprana. Para análisis causales reales revisar antes si esta regla es apropiada y mantener un registro de exclusiones.

In [ ]:
valid_rows = valid_group & valid_page & valid_convert & valid_user & valid_time & assignment_match
eligible = df.loc[valid_rows].copy()

eligible = eligible.sort_values(["user_id", "timestamp"], kind="mergesort")
df_clean = eligible.drop_duplicates("user_id", keep="first").reset_index(drop=True)
df_clean["converted"] = df_clean["converted"].astype("int8")

cleaning_summary = pd.DataFrame({
    "metric": ["raw_rows", "invalid_or_mismatch_rows", "valid_rows_before_dedup", "duplicate_valid_user_rows_removed", "clean_rows", "unique_clean_users"],
    "value": [len(df), int((~valid_rows).sum()), len(eligible), len(eligible) - len(df_clean), len(df_clean), df_clean["user_id"].nunique()]
})
display(cleaning_summary)

assert df_clean["user_id"].is_unique
assert df_clean["converted"].isin([0,1]).all()
assert df_clean["timestamp"].notna().all()
assert (
    ((df_clean["group"] == "control") & (df_clean["landing_page"] == "old_page")) |
    ((df_clean["group"] == "treatment") & (df_clean["landing_page"] == "new_page"))
).all()

## 8. EDA sobre la muestra limpia

In [ ]:
eda_summary = (
    df_clean.groupby("group", observed=True)["converted"]
    .agg(users="count", conversions="sum", conversion_rate="mean")
    .reindex(["control", "treatment"])
)
eda_summary["non_conversions"] = eda_summary["users"] - eda_summary["conversions"]
eda_summary["conversion_rate_pct"] = (100 * eda_summary["conversion_rate"]).round(3)
display(eda_summary)

n_total = int(eda_summary["users"].sum())
print("Participación por grupo (%):")
display((eda_summary["users"] / n_total * 100).round(2))

rates = eda_summary["conversion_rate_pct"]
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(rates.index, rates.values)
ax.set(xlabel="Grupo", ylabel="Tasa de conversión (%)", title="Conversión observada por grupo")
ax.set_ylim(0, max(rates.values)*1.2)
for i,v in enumerate(rates.values):
    ax.text(i, v + max(rates.values)*0.03, f"{v:.2f}%", ha="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conversion_by_group.png", dpi=160)
plt.show()

## 9. EDA temporal
Los conteos por fecha revelan cambios de tráfico, experimentos interrumpidos y problemas de instrumentación. Una tendencia visual no demuestra un efecto causal.

In [ ]:
df_clean["date"] = df_clean["timestamp"].dt.date
by_day = df_clean.groupby(["date", "group"], observed=True).agg(
    users=("user_id", "size"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean")
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for group, d in by_day.groupby("group"):
    axes[0].plot(d["date"], d["users"], label=group)
    axes[1].plot(d["date"], 100*d["conversion_rate"], label=group)
axes[0].set(ylabel="Usuarios", title="Tráfico diario por grupo")
axes[1].set(xlabel="Fecha (UTC)", ylabel="Conversión (%)", title="Conversión diaria por grupo")
for ax in axes: ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "daily_trends.png", dpi=160)
plt.show()

display(by_day.head())

## 10. Comprobaciones finales y exportación
Los datos limpios son descriptivos, no la conclusión estadística del experimento. En el siguiente notebook se estudiarán las distribuciones y, después, se construirá la prueba de hipótesis.

In [ ]:
df_clean.to_csv(OUTPUT_DIR / "ab_data_clean.csv", index=False)
eda_summary.reset_index().to_csv(OUTPUT_DIR / "eda_summary.csv", index=False)
quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)
cleaning_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
by_day.to_csv(OUTPUT_DIR / "daily_conversions.csv", index=False)

if USE_S3:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            prefix = "processed" if path.name == "ab_data_clean.csv" else "results"
            dest_key = f"{prefix}/{path.name}"
            s3.upload_file(str(path), BUCKET, dest_key)
            print(f"Subido: s3://{BUCKET}/{dest_key}")
else:
    print("Archivos guardados localmente en:", OUTPUT_DIR.resolve())

## 11. Preguntas de interpretación

1. ¿La proporción de tráfico en cada grupo se acerca a lo esperado? ¿Se había planificado una asignación 50/50?
2. ¿Cuántos usuarios aparecen más de una vez? ¿Pueden sesgar los errores estándar?
3. ¿Qué porcentaje de registros tiene incongruencias grupo-página? ¿Es similar en ambos grupos?
4. ¿Qué tan diferentes son las tasas descriptivas de conversión? ¿Por qué todavía no podemos concluir significancia?
5. ¿Existe algún día de tráfico o conversión anómalo que amerite investigar?
6. ¿Qué cambiaría si se hiciera un análisis *intention-to-treat* en lugar de excluir exposiciones incorrectas?

**Nota:** evita ajustar las reglas de exclusión después de ver qué variante ganó.